# Docscan — Exploratory Data Analysis

**Data:** `data/train`, `data/val`, `data/test` — each a Hugging Face `datasets`
dump (`load_from_disk`), with an `image` column (PIL image) and a `label`
column (`ClassLabel`, 16 document types).

**What this notebook checks, and why:**

1. Dataset overview — do the split sizes look sane?
2. Class distribution & imbalance — are some document types much rarer than others?
3. Split consistency — do train/val/test hold the same class *proportions*?
4. Image dimensions & aspect ratio — how much do you need to resize/pad?
5. Data quality — corrupted files, near-blank scans, near-duplicate images.
6. Deeper visual exploration — what do the classes actually look like, side by side?
7. Summary — the numbers that matter for modeling decisions.

**Requirements:** `pillow`, `pandas`, `matplotlib`, `datasets` (numpy comes along
as a dependency of pandas/matplotlib). To open this file you also need Jupyter
(`pip install jupyterlab`) or an editor with notebook support (e.g. VS Code).

```bash
python -m pip install pillow pandas matplotlib "datasets==5.0.1" jupyterlab
jupyter lab eda_notebook.ipynb
```


In [ ]:
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from datasets import load_from_disk
from PIL import Image

%matplotlib inline

# ---------------------------------------------------------------
# Chart style — a small validated palette (colorblind-checked), not
# an arbitrary matplotlib default. Categorical slots are used only
# where color encodes identity (train/val/test); everything that
# encodes plain magnitude (counts, histograms, heatmaps) uses a
# single hue so color never implies "different kind of thing".
# ---------------------------------------------------------------
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
CRITICAL = "#d03b3b"
INK, INK_SECONDARY = "#0b0b0b", "#52514e"
GRID, BASELINE, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"

SPLIT_COLORS = {"train": BLUE, "val": ORANGE, "test": AQUA}

_seq_steps = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
              "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
SEQ_BLUE = LinearSegmentedColormap.from_list("seq_blue", _seq_steps)

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": BASELINE,
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": INK_SECONDARY,
    "ytick.color": INK_SECONDARY,
    "grid.color": GRID,
    "axes.grid": True,
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "figure.dpi": 100,
})

RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)


## Load the data

In [ ]:
DATA_DIR = Path("data")
SPLITS = ["train", "val", "test"]

# How many images per split to *decode* for the pixel-level sections (4-6):
# dimensions, brightness, blank/duplicate checks. Full dataset works too,
# just slower — raise this (or set to None) once you trust the pipeline.
SAMPLE_SIZE = 3000

print("Loading dataset splits from disk...")
datasets = {split: load_from_disk(str(DATA_DIR / split)) for split in SPLITS}
class_names = datasets["train"].features["label"].names
n_classes = len(class_names)
print(f"Loaded {n_classes} classes: {class_names}")


def get_subset(ds, n):
    """Deterministic sample of a split, or the whole thing if n is None."""
    if n is None or n >= len(ds):
        return ds
    return ds.shuffle(seed=RNG_SEED).select(range(n))


## 1. Dataset overview

How many images does each split actually contain, and does the train/val/test
split look like a sane division (e.g. roughly 70/15/15, 80/10/10)?


In [ ]:
overview = pd.DataFrame({
    "split": SPLITS,
    "images": [len(datasets[s]) for s in SPLITS],
})
overview["share_%"] = (overview["images"] / overview["images"].sum() * 100).round(1)
overview.loc[len(overview)] = ["total", overview["images"].sum(), 100.0]
overview


## 2. Class distribution & imbalance

Are the 16 document types represented evenly, or are some classes much rarer
than others? A skewed training set biases a classifier toward the majority
classes unless you weight the loss, resample, or pick a metric that accounts
for it (e.g. macro-F1 instead of raw accuracy).


In [ ]:
labels_by_split = {split: datasets[split]["label"] for split in SPLITS}

distribution = {split: dict(Counter(labels_by_split[split])) for split in SPLITS}
dist_df = pd.DataFrame(distribution).reindex(range(n_classes)).fillna(0).astype(int)
dist_df.index = class_names
dist_df = dist_df[SPLITS]
dist_df


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(n_classes)
width = 0.26
for i, split in enumerate(SPLITS):
    ax.bar(x + (i - 1) * width, dist_df[split], width, label=split, color=SPLIT_COLORS[split])
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_ylabel("Number of images")
ax.set_title("Images per class, by split")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
train_counts = dist_df["train"]
imbalance_ratio = train_counts.max() / train_counts.min()
print(f"Most represented class:  {train_counts.idxmax()} ({train_counts.max()} images)")
print(f"Least represented class: {train_counts.idxmin()} ({train_counts.min()} images)")
print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}x")


## 3. Split consistency check

A properly stratified split keeps roughly the same *proportion* of each class
in train, val, and test. If a class's share drifts a lot between splits, your
val/test metrics for that class become less trustworthy (e.g. a class that's
5% of train but 15% of test).


In [ ]:
prop_df = dist_df.div(dist_df.sum(axis=0), axis=1) * 100  # % within each split

fig, ax = plt.subplots(figsize=(6, 8))
im = ax.imshow(prop_df.values, cmap=SEQ_BLUE, aspect="auto")
ax.set_xticks(range(len(SPLITS)))
ax.set_xticklabels(SPLITS)
ax.set_yticks(range(n_classes))
ax.set_yticklabels(class_names)
ax.set_title("Class share within each split (%)")
ax.grid(False)
for i in range(n_classes):
    for j in range(len(SPLITS)):
        val = prop_df.values[i, j]
        r, g, b, _ = im.cmap(im.norm(val))
        luminance = 0.299 * r + 0.587 * g + 0.114 * b
        ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=8,
                color=INK if luminance > 0.6 else "#ffffff")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("% of split")
plt.tight_layout()
plt.show()


In [ ]:
deviation = (prop_df.max(axis=1) - prop_df.min(axis=1)).sort_values(ascending=False)
print("Largest train/val/test share gaps (percentage points):")
deviation.head(10).round(2)


## 4. Image dimensions, quality & duplicates

One decode pass over a sample of each split — this gathers everything the
rest of the notebook needs: width/height/aspect ratio, color mode, grayscale
brightness (for the blank-scan check in section 5), and a perceptual hash
(for the duplicate check in section 5).


In [ ]:
def average_hash(img, hash_size=8):
    """A cheap perceptual hash: resize small, threshold vs. the mean.
    Near-identical images land on the same hash; it's a similarity signal,
    not a cryptographic one."""
    small = img.convert("L").resize((hash_size, hash_size), Image.LANCZOS)
    arr = np.asarray(small, dtype=np.float64)
    bits = (arr > arr.mean()).flatten()
    h = 0
    for bit in bits:
        h = (h << 1) | int(bit)
    return h


image_rows = []
bad_images = []
hash_to_locations = {}

for split in SPLITS:
    subset = get_subset(datasets[split], SAMPLE_SIZE)
    for idx in range(len(subset)):
        row = subset[idx]
        try:
            img = row["image"]
            w, h = img.size
            gray = np.asarray(img.convert("L"), dtype=np.float64)
            ahash = average_hash(img)
            cls = class_names[row["label"]]
            hash_to_locations.setdefault(ahash, []).append((split, idx, cls))
            image_rows.append({
                "split": split,
                "class": cls,
                "row_index": idx,
                "width": w,
                "height": h,
                "aspect_ratio": round(w / h, 3),
                "mode": img.mode,
                "format": img.format,
                "gray_mean": gray.mean(),
                "gray_std": gray.std(),
                "ahash": ahash,
            })
        except Exception as e:
            bad_images.append((split, idx, str(e)))

image_df = pd.DataFrame(image_rows)
print(f"Decoded {len(image_df)} images ({len(bad_images)} failed to decode)")


def fetch_image(row):
    """Re-fetch the actual PIL image for a row of image_df (deterministic re-sample)."""
    subset = get_subset(datasets[row["split"]], SAMPLE_SIZE)
    return subset[int(row["row_index"])]["image"]


image_df.describe(include="all")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(
    axes,
    ["width", "height", "aspect_ratio"],
    ["Width (px)", "Height (px)", "Aspect ratio (w / h)"],
):
    ax.hist(image_df[col], bins=40, color=BLUE, edgecolor=SURFACE)
    ax.set_title(title)
    ax.set_xlabel(title)
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, ["mode", "format"]):
    counts = image_df[col].astype(str).value_counts()
    ax.barh(counts.index, counts.values, color=BLUE)
    ax.set_title(f"Image {col}")
    ax.set_xlabel("Count")
    ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 5. Data quality checks

### 5a. Corrupted / undecodable images


In [ ]:
if bad_images:
    print(f"{len(bad_images)} images failed to decode:")
    for split, idx, err in bad_images[:20]:
        print(f"  {split}[{idx}]: {err}")
else:
    print("No corrupted images found in the sample.")


### 5b. Near-blank / low-content scans

Heuristic, not a certainty: flags pages that are almost entirely white (mean
grayscale value very high) or almost entirely flat/uniform (very low pixel
variance — a blank or mis-scanned sheet). Worth a manual glance at the
flagged examples below, not an automatic delete — a genuinely sparse
document (e.g. a short memo) can also trip this.


In [ ]:
BLANK_MEAN_THRESHOLD = 250  # near-white
BLANK_STD_THRESHOLD = 8     # almost no pixel variation

image_df["possibly_blank"] = (
    (image_df["gray_mean"] > BLANK_MEAN_THRESHOLD)
    | (image_df["gray_std"] < BLANK_STD_THRESHOLD)
)

n_blank = int(image_df["possibly_blank"].sum())
print(f"{n_blank} of {len(image_df)} sampled images "
      f"({image_df['possibly_blank'].mean() * 100:.1f}%) look possibly blank/low-content")

blank_by_class = image_df.groupby("class")["possibly_blank"].sum().sort_values(ascending=False)
blank_by_class[blank_by_class > 0]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(image_df["gray_mean"], bins=50, color=BLUE, edgecolor=SURFACE)
ax.axvline(BLANK_MEAN_THRESHOLD, color=CRITICAL, linestyle="--", linewidth=2,
           label=f"blank threshold ({BLANK_MEAN_THRESHOLD})")
ax.set_title("Mean grayscale brightness per image")
ax.set_xlabel("Mean pixel value (0 = black, 255 = white)")
ax.set_ylabel("Count")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
flagged = image_df[image_df["possibly_blank"]]
sample_n = min(6, len(flagged))
if sample_n:
    flagged_sample = flagged.sample(sample_n, random_state=RNG_SEED)
    fig, axes = plt.subplots(1, sample_n, figsize=(15, 3))
    if sample_n == 1:
        axes = [axes]
    for ax, (_, r) in zip(axes, flagged_sample.iterrows()):
        img = fetch_image(r)
        ax.imshow(img, cmap="gray" if img.mode == "L" else None)
        ax.set_title(r["class"], fontsize=9)
        ax.axis("off")
    plt.suptitle("Flagged as possibly blank/low-content")
    plt.tight_layout()
    plt.show()
else:
    print("Nothing flagged — nothing to show.")


### 5c. Near-duplicate images

Perceptual-hash groups with more than one sampled image. This catches
near-identical scans (the same document appearing more than once), not just
byte-identical files — and it will also cluster genuinely blank pages
together, since they all *look* alike. Cross-check against the blank-page
flag above before treating a group as a true duplicate.


In [ ]:
dup_groups = {h: locs for h, locs in hash_to_locations.items() if len(locs) > 1}
n_dup_images = sum(len(v) for v in dup_groups.values())
print(f"{len(dup_groups)} hash groups contain more than one sampled image "
      f"({n_dup_images} images total)")

sorted_groups = sorted(dup_groups.items(), key=lambda kv: -len(kv[1]))
for h, locs in sorted_groups[:5]:
    classes_involved = sorted(set(c for _, _, c in locs))
    print(f"hash {h:016x}: {len(locs)} images -> classes: {classes_involved}")


In [ ]:
if sorted_groups:
    h, locs = sorted_groups[0]
    examples = locs[:6]
    fig, axes = plt.subplots(1, len(examples), figsize=(15, 3))
    if len(examples) == 1:
        axes = [axes]
    for ax, (split, idx, cls) in zip(axes, examples):
        subset = get_subset(datasets[split], SAMPLE_SIZE)
        img = subset[idx]["image"]
        ax.imshow(img, cmap="gray" if img.mode == "L" else None)
        ax.set_title(f"{split}/{cls}", fontsize=9)
        ax.axis("off")
    plt.suptitle(f"Largest near-duplicate group ({len(locs)} images)")
    plt.tight_layout()
    plt.show()
else:
    print("No near-duplicate groups found in the sample.")


## 6. Deeper visual exploration

What do the classes actually look like, and do they differ systematically in
shape or tone (not just label)? Small multiples (one panel per class) instead
of overlaying 16 classes in a single plot — with this many categories,
overlaid colors stop being distinguishable anyway.


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 16), sharex=True, sharey=True)
xmax = image_df["width"].max() * 1.05
ymax = image_df["height"].max() * 1.05
for ax, cls in zip(axes.flat, class_names):
    sub = image_df[image_df["class"] == cls]
    ax.scatter(sub["width"], sub["height"], s=8, alpha=0.35, color=BLUE, linewidths=0)
    ax.set_title(cls, fontsize=9)
    ax.set_xlim(0, xmax)
    ax.set_ylim(0, ymax)
fig.supxlabel("Width (px)")
fig.supylabel("Height (px)")
fig.suptitle("Image width vs. height, by class", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12), sharex=True, sharey=True)
for ax, cls in zip(axes.flat, class_names):
    sub = image_df[image_df["class"] == cls]
    ax.hist(sub["gray_mean"], bins=20, color=BLUE, edgecolor=SURFACE)
    ax.set_title(cls, fontsize=9)
fig.supxlabel("Mean grayscale brightness")
fig.supylabel("Count")
fig.suptitle("Brightness distribution, by class", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for ax, cls in zip(axes.flat, class_names):
    row = image_df[image_df["class"] == cls].sample(1, random_state=RNG_SEED).iloc[0]
    img = fetch_image(row)
    ax.imshow(img, cmap="gray" if img.mode == "L" else None)
    ax.set_title(cls, fontsize=10)
    ax.axis("off")
plt.suptitle("One example per class", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
def show_class_examples(class_name, n=6):
    """Look at n random sampled examples of one class. Call this yourself
    with any name from `class_names` to dig into a specific class."""
    sub = image_df[image_df["class"] == class_name]
    sample = sub.sample(min(n, len(sub)), random_state=RNG_SEED)
    fig, axes = plt.subplots(1, len(sample), figsize=(15, 4))
    if len(sample) == 1:
        axes = [axes]
    for ax, (_, r) in zip(axes, sample.iterrows()):
        img = fetch_image(r)
        ax.imshow(img, cmap="gray" if img.mode == "L" else None)
        ax.set_title(f"{r['width']}x{r['height']}", fontsize=9)
        ax.axis("off")
    plt.suptitle(f"Examples: {class_name}")
    plt.tight_layout()
    plt.show()


show_class_examples("invoice")
show_class_examples("handwritten")


## 7. Summary

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print("Splits: " + ", ".join(f"{s}={len(datasets[s])}" for s in SPLITS))
print(f"Classes: {n_classes}")
print(f"Train imbalance ratio (max/min class): {imbalance_ratio:.2f}x "
      f"({train_counts.idxmax()} vs {train_counts.idxmin()})")
print(f"Largest train/val/test share gap: {deviation.index[0]} ({deviation.iloc[0]:.2f} pp)")
print(f"Sampled for pixel-level checks: {len(image_df)} images "
      f"({'all' if SAMPLE_SIZE is None else SAMPLE_SIZE} per split)")
print(f"Decode failures: {len(bad_images)}")
print(f"Possibly-blank images: {n_blank} ({image_df['possibly_blank'].mean() * 100:.1f}%)")
print(f"Near-duplicate groups: {len(dup_groups)} ({n_dup_images} images)")
print(f"Width range: {image_df['width'].min()}-{image_df['width'].max()} px")
print(f"Height range: {image_df['height'].min()}-{image_df['height'].max()} px")
